# Proyecto 1 - Hoppers con búsqueda adversarial

**CC3085 - Inteligencia Artificial**

## Parte 1: representación e interpretación del juego

Este notebook seguirá el mismo orden utilizado en el laboratorio de Tic-Tac-Toe y en
Lecture 0. Primero se define el juego mediante funciones independientes del agente:

| Función | Responsabilidad |
|---|---|
| `initial_state()` | Construir el tablero inicial de Hoppers. |
| `player(state)` | Indicar a qué jugador le corresponde mover. |
| `actions(state)` | Obtener todas las jugadas legales. |
| `result(state, action)` | Crear el estado posterior sin modificar el original. |
| `winner(state)` | Identificar al ganador, si existe. |
| `terminal(state)` | Indicar si la partida terminó. |
| `utility(state)` | Asignar `+1` a P1, `-1` a P2 y `0` si aún no hay ganador. |

En etapas posteriores se agregarán Minimax con poda alfa-beta, límite de profundidad,
heurística, control de tiempo e interfaz gráfica.


## 1. Reglas que representará el programa

- El tablero tiene 10 filas y 10 columnas.
- Cada jugador inicia con 15 piezas dentro de un campamento triangular.
- P1 comienza en la esquina superior izquierda y busca llegar a la esquina inferior derecha.
- P2 comienza en la esquina inferior derecha y busca llegar a la esquina superior izquierda.
- Una pieza puede dar un **paso** a cualquier casilla vecina vacía, incluyendo diagonales.
- También puede **saltar** una pieza propia o rival hacia la casilla vacía inmediatamente posterior.
- En la misma jugada se pueden encadenar varios saltos. Las piezas saltadas no se capturan.
- Un paso no puede combinarse con saltos dentro de la misma jugada.
- Durante una cadena no se permite repetir una casilla de aterrizaje. Esto evita ciclos.
- Se aplica la regla anti-bloqueo: un campamento objetivo lleno cuenta como completado aunque
  contenga alguna pieza del jugador que inició allí. Se exige al menos una pieza del jugador
  que está llegando para que nadie gane en el estado inicial.


In [2]:
from dataclasses import dataclass
from typing import Optional

BOARD_SIZE = 10
P1 = "P1"  # En la interfaz será la BYD TI-7, me gusta ese vehículo.
P2 = "P2"  # En la interfaz será la Land Cruiser Prado
EMPTY = None

Position = tuple[int, int]
Action = tuple[Position, ...]
Board = tuple[tuple[Optional[str], ...], ...]

# Campamento superior izquierdo: 5 + 4 + 3 + 2 + 1 = 15 casillas.
"""
-----
----
---
--
-  
"""
P1_CAMP = frozenset(
    (row, col)
    for row in range(5)
    for col in range(5 - row)
)

# El campamento de P2 es el reflejo del campamento de P1.
P2_CAMP = frozenset(
    (BOARD_SIZE - 1 - row, BOARD_SIZE - 1 - col)
    for row, col in P1_CAMP
)

# Movimiento horizontal, vertical y diagonal.
DIRECTIONS = tuple(
    (dr, dc)
    for dr in (-1, 0, 1)
    for dc in (-1, 0, 1)
    if (dr, dc) != (0, 0)
)


@dataclass(frozen=True)
class State:
    """Estado completo: tablero inmutable y jugador al que le toca mover."""

    board: Board
    turn: str


### ¿Por qué el turno forma parte del estado?

En Tic-Tac-Toe se podía contar cuántas X y O había. En Hoppers las piezas no se eliminan,
por lo que siempre hay 15 piezas de cada jugador. El tablero por sí solo no permite conocer
el turno. Guardarlo explícitamente también hace que cada nodo del árbol de búsqueda contenga
toda la información necesaria.

El tablero y las acciones usan tuplas para que sean inmutables y, más adelante, puedan
usarse como llaves de una caché durante Minimax.


In [3]:
def initial_state() -> State:
    """Devuelve el estado inicial de Hoppers; P1 siempre mueve primero."""
    board = [[EMPTY for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]

    for row, col in P1_CAMP:
        board[row][col] = P1

    for row, col in P2_CAMP:
        board[row][col] = P2

    
    return State(
        board=tuple(tuple(row) for row in board),
        turn=P1,
    )


def player(state: State) -> str:
    """Devuelve el jugador al que le corresponde mover."""
    return state.turn


def other_player(current_player: str) -> str:
    """Devuelve el contrincante de current_player."""
    return P2 if current_player == P1 else P1


In [4]:
# auxiliar de pruebas
def check(label, condition):
    print(("PASS  " if condition is True else "MULA ") + label)
    return bool(condition)


_initial = initial_state()
check("el tablero es de 10 x 10", len(_initial.board) == 10 and all(len(row) == 10 for row in _initial.board))
check("P1 inicia con 15 piezas", sum(cell == P1 for row in _initial.board for cell in row) == 15)
check("P2 inicia con 15 piezas", sum(cell == P2 for row in _initial.board for cell in row) == 15)
check("P1 mueve primero", player(_initial) == P1)
check("cada campamento tiene 15 casillas", len(P1_CAMP) == len(P2_CAMP) == 15)


PASS  el tablero es de 10 x 10
PASS  P1 inicia con 15 piezas
PASS  P2 inicia con 15 piezas
PASS  P1 mueve primero
PASS  cada campamento tiene 15 casillas


True

## 2. `actions(state)`: pasos y saltos encadenados

Una acción guarda la ruta completa recorrida por una pieza. Sus dos primeras coordenadas
siempre representan origen y primer destino. Si existen más coordenadas, son los siguientes
aterrizajes de una cadena de saltos.

Para encontrar todas las cadenas se usa una búsqueda en profundidad pequeña que parte de
cada pieza del jugador. Cada prefijo legal también es una acción válida, porque el jugador
puede decidir detenerse después de cualquier salto.


In [5]:
def inside_board(position: Position) -> bool:
    """Indica si una coordenada pertenece al tablero."""
    row, col = position
    return 0 <= row < BOARD_SIZE and 0 <= col < BOARD_SIZE


def _jump_actions_from(state: State, origin: Position) -> set[Action]:
    """Encuentra todas las rutas de uno o más saltos desde origin."""
    board = [list(row) for row in state.board]
    piece = board[origin[0]][origin[1]]
    jump_actions = set()

    def explore(current: Position, path: Action, visited: frozenset[Position]):
        current_row, current_col = current

        for dr, dc in DIRECTIONS:
            middle = (current_row + dr, current_col + dc)
            landing = (current_row + 2 * dr, current_col + 2 * dc)

            if not inside_board(middle) or not inside_board(landing):
                continue

            middle_piece = board[middle[0]][middle[1]]
            landing_piece = board[landing[0]][landing[1]]

            if middle_piece is EMPTY or landing_piece is not EMPTY or landing in visited:
                continue

            # Se simula el salto para que el siguiente salto vea el tablero correcto.
            board[current_row][current_col] = EMPTY
            board[landing[0]][landing[1]] = piece

            new_path = path + (landing,)
            jump_actions.add(new_path)
            explore(landing, new_path, visited | {landing})

            # Deshacer permite explorar otra rama sin contaminarla.
            board[landing[0]][landing[1]] = EMPTY
            board[current_row][current_col] = piece

    explore(origin, (origin,), frozenset({origin}))
    return jump_actions


def actions(state: State) -> set[Action]:
    """Devuelve todos los pasos y todas las rutas de saltos legales."""
    if terminal(state):
        return set()

    legal_actions = set()

    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if state.board[row][col] != player(state):
                continue

            origin = (row, col)

            # Pasos de una casilla.
            for dr, dc in DIRECTIONS: 
                destination = (row + dr, col + dc)
                if inside_board(destination) and state.board[destination[0]][destination[1]] is EMPTY:
                    legal_actions.add((origin, destination))

            # Saltos simples y encadenados.
            legal_actions.update(_jump_actions_from(state, origin))

    return legal_actions


> Nota de diseño: `actions` llama a `terminal`, (se define más abajo).
 Esto es válido en Python porque la llamada ocurre al ejecutar la función, después de haber corrido todas las 
 celdas de definiciones..


## 3. `result(state, action)`: aplicar sin mutar

Minimax examinará muchas jugadas hipotéticas desde un mismo estado. Si `result` modificara
el tablero recibido, una rama alteraría las demás. Por eso se crea una copia, se valida la
acción contra `actions(state)` y se devuelve un `State` nuevo con el turno alternado.


In [6]:
def result(state: State, action: Action) -> State:
    """Devuelve el estado posterior a una acción legal sin modificar state."""
    if action not in actions(state):
        raise ValueError(f"Jugada ilegal: {action}")

    origin = action[0]
    destination = action[-1]
    board = [list(row) for row in state.board]

    board[origin[0]][origin[1]] = EMPTY
    board[destination[0]][destination[1]] = player(state)

    return State(
        board=tuple(tuple(row) for row in board),
        turn=other_player(player(state)),
    )


## 4. `winner`, `terminal` y `utility`

P1 intenta completar `P2_CAMP` y P2 intenta completar `P1_CAMP`. Con la regla anti-bloqueo,
un campamento se considera completo si todas sus casillas están ocupadas y al menos una
pertenece al jugador que está llegando. Las piezas rivales que quedaron bloqueando también
cuentan como ocupación.

Si una posición construida artificialmente hace que ambos campamentos estén completos,
se considera ganador al jugador que realizó la última jugada, es decir, al contrario de
`state.turn`.


In [7]:
def _completed_target_camp(state: State, candidate: str) -> bool:
    """Comprueba la meta con la regla anti-bloqueo elegida para el proyecto."""
    target_camp = P2_CAMP if candidate == P1 else P1_CAMP
    occupants = [state.board[row][col] for row, col in target_camp]

    return all(piece is not EMPTY for piece in occupants) and candidate in occupants


def winner(state: State) -> Optional[str]:
    """Devuelve P1, P2 o None si todavía no hay ganador."""
    previous_player = other_player(player(state))

    # Revisar primero a quien acaba de mover respeta la idea de "primero en completar".
    if _completed_target_camp(state, previous_player):
        return previous_player

    if _completed_target_camp(state, player(state)):
        return player(state)

    return None


def terminal(state: State) -> bool:
    """La partida termina cuando alguno de los jugadores completa su meta."""
    return winner(state) is not None


def utility(state: State) -> int:
    """Asigna +1 si gana P1, -1 si gana P2 y 0 si no existe ganador."""
    game_winner = winner(state)

    if game_winner == P1:
        return 1
    if game_winner == P2:
        return -1
    return 0


## 5. Pruebas de la Parte 1

Una de las buenas prácticas que aprendí haciendo los laboratorios en la clase de IA. Estas pruebas verifican los casos centrales de la rúbrica, que son el tablero y que los turnos sean los correctos. También se verifican los movimientos en ocho direcciones, salto sobre cualquiera de los dos colores, saltos
encadenados, ausencia de mutación, cambio de turno, jugadas inválidas y estados de victoria.


In [8]:
def state_from_pieces(p1_positions=(), p2_positions=(), turn=P1):
    """Auxiliar exclusivo para construir escenarios pequeños de prueba."""
    board = [[EMPTY for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]
    for row, col in p1_positions:
        board[row][col] = P1
    for row, col in p2_positions:
        board[row][col] = P2
    return State(tuple(tuple(row) for row in board), turn)


# Pasos en las ocho direcciones desde el centro.
step_state = state_from_pieces(p1_positions={(5, 5)})
expected_steps = {
    ((5, 5), (5 + dr, 5 + dc))
    for dr, dc in DIRECTIONS
}
check("una pieza central puede dar 8 pasos", expected_steps <= actions(step_state))

# Se puede saltar una pieza propia y una rival.
own_jump_state = state_from_pieces(p1_positions={(2, 2), (3, 3)})
rival_jump_state = state_from_pieces(p1_positions={(2, 2)}, p2_positions={(3, 3)})
check("se puede saltar una pieza propia", ((2, 2), (4, 4)) in actions(own_jump_state))
check("se puede saltar una pieza rival", ((2, 2), (4, 4)) in actions(rival_jump_state))

# Cadena: (0,0) salta (1,1), aterriza en (2,2), salta (3,3) y llega a (4,4).
chain_state = state_from_pieces(
    p1_positions={(0, 0), (1, 1)},
    p2_positions={(3, 3)},
)
chain = ((0, 0), (2, 2), (4, 4))
check("el salto simple de una cadena también es legal", ((0, 0), (2, 2)) in actions(chain_state))
check("se generan saltos encadenados", chain in actions(chain_state))

# result crea otro estado y conserva intacto el anterior.
after_chain = result(chain_state, chain)
check("result no muta el estado original", chain_state.board[0][0] == P1 and chain_state.board[4][4] is EMPTY)
check("result mueve la pieza al destino final", after_chain.board[0][0] is EMPTY and after_chain.board[4][4] == P1)
check("las piezas saltadas no se capturan", after_chain.board[1][1] == P1 and after_chain.board[3][3] == P2)
check("result cambia el turno", player(after_chain) == P2)

try:
    result(chain_state, ((0, 0), (0, 3)))
    invalid_action_rejected = False
except ValueError:
    invalid_action_rejected = True
check("result rechaza jugadas ilegales", invalid_action_rejected)

# El estado inicial no puede activar accidentalmente la regla anti-bloqueo.
check("el estado inicial no tiene ganador", winner(initial_state()) is None)
check("el estado inicial no es terminal", terminal(initial_state()) is False)

# P1 llena el campamento derecho, excepto una casilla que P2 sigue bloqueando.
p2_target = sorted(P2_CAMP)
anti_block_state = state_from_pieces(
    p1_positions=set(p2_target[:-1]),
    p2_positions={p2_target[-1]},
    turn=P2,
)
check("la regla anti-bloqueo reconoce la victoria de P1", winner(anti_block_state) == P1)
check("un estado ganador es terminal", terminal(anti_block_state) is True)
check("la victoria de P1 tiene utilidad +1", utility(anti_block_state) == 1)

# Caso simétrico para P2.
p1_target = sorted(P1_CAMP)
p2_win_state = state_from_pieces(
    p1_positions={p1_target[-1]},
    p2_positions=set(p1_target[:-1]),
    turn=P1,
)
check("la regla anti-bloqueo reconoce la victoria de P2", winner(p2_win_state) == P2)
check("la victoria de P2 tiene utilidad -1", utility(p2_win_state) == -1)


PASS  una pieza central puede dar 8 pasos
PASS  se puede saltar una pieza propia
PASS  se puede saltar una pieza rival
PASS  el salto simple de una cadena también es legal
PASS  se generan saltos encadenados
PASS  result no muta el estado original
PASS  result mueve la pieza al destino final
PASS  las piezas saltadas no se capturan
PASS  result cambia el turno
PASS  result rechaza jugadas ilegales
PASS  el estado inicial no tiene ganador
PASS  el estado inicial no es terminal
PASS  la regla anti-bloqueo reconoce la victoria de P1
PASS  un estado ganador es terminal
PASS  la victoria de P1 tiene utilidad +1
PASS  la regla anti-bloqueo reconoce la victoria de P2
PASS  la victoria de P2 tiene utilidad -1


True

NÍTIDOOOO
# SEGUNDA ETAPA
## MINIMAX CON PODA ALFA-BETA



In [9]:

@dataclass
class SearchStats:
    """Contadores para observar el trabajo realizado por Minimax."""
    visited_nodes: int = 0
    pruned_branches: int = 0

In [10]:
def _state_after_legal_action(state, action):
    """
    Aplica una acción que ya fue obtenida mediante actions(state).
    No necesita volver a comprobar si es legal.
    """
    origin = action[0]
    destination = action[-1]

    board = [list(row) for row in state.board]

    board[origin[0]][origin[1]] = EMPTY
    board[destination[0]][destination[1]] = player(state)

    return State(
        board=tuple(tuple(row) for row in board),
        turn=other_player(player(state))
    )

In [11]:
def _ordered_successors(state):
    """
    Devuelve pares (acción, nuevo_estado).
    Las victorias inmediatas se colocan primero.
    """
    successors = []

    for action in actions(state):
        child = _state_after_legal_action(state, action)

        immediate_win = winner(child) == player(state)
        priority = 0 if immediate_win else 1

        successors.append((priority, action, child))

    successors.sort(key=lambda item: (item[0], item[1]))

    return [
        (action, child)
        for _, action, child in successors
    ]

### Función MAX 
Donde p1 trata de obtener el valor más grande (como en el diagrma del árbol).

In [12]:
def _max_value_ab(state, alpha, beta, stats):
    """Devuelve el mejor valor que P1 puede garantizar."""
    stats.visited_nodes += 1

    if terminal(state):
        return utility(state)

    value = float("-inf")

    for _, child in _ordered_successors(state):
        value = max(
            value,
            _min_value_ab(child, alpha, beta, stats)
        )

        # Poda beta
        if value >= beta:
            stats.pruned_branches += 1
            return value

        alpha = max(alpha, value)

    return value

In [13]:
def _min_value_ab(state, alpha, beta, stats):
    """Devuelve el mejor valor que P2 puede garantizar."""
    stats.visited_nodes += 1

    if terminal(state):
        return utility(state)

    value = float("inf")

    for _, child in _ordered_successors(state):
        value = min(
            value,
            _max_value_ab(child, alpha, beta, stats)
        )

        # Poda alfa
        if value <= alpha:
            stats.pruned_branches += 1
            return value

        beta = min(beta, value)

    return value

### Mejor jugada al podar

In [14]:
def minimax_alpha_beta(state, return_stats=False):
    """
    Devuelve una jugada óptima usando Minimax con poda alfa-beta.

    Si return_stats=True, también devuelve las estadísticas
    de la búsqueda.
    """
    if terminal(state):
        if return_stats:
            return None, SearchStats()

        return None

    stats = SearchStats()
    best_action = None

    # P1 funciona como MAX
    if player(state) == P1:
        best_value = float("-inf")
        alpha = float("-inf")
        beta = float("inf")

        for action, child in _ordered_successors(state):
            value = _min_value_ab(
                child,
                alpha,
                beta,
                stats
            )

            if value > best_value:
                best_value = value
                best_action = action

            alpha = max(alpha, best_value)

            # +1 es el mejor resultado posible para P1
            if best_value == 1:
                break

    # P2 funciona como MIN
    else:
        best_value = float("inf")
        alpha = float("-inf")
        beta = float("inf")

        for action, child in _ordered_successors(state):
            value = _max_value_ab(
                child,
                alpha,
                beta,
                stats
            )

            if value < best_value:
                best_value = value
                best_action = action

            beta = min(beta, best_value)

            # -1 es el mejor resultado posible para P2
            if best_value == -1:
                break

    if return_stats:
        return best_action, stats

    return best_action

In [15]:
def minimax_agent(state):
    """Agente con formato state -> action."""
    return minimax_alpha_beta(state)

Uso

In [16]:
def one_move_from_winning(candidate):
    """
    Construye un estado donde candidate puede ganar
    realizando una sola jugada.
    """
    target = P2_CAMP if candidate == P1 else P1_CAMP

    for empty_position in sorted(target):
        empty_row, empty_col = empty_position

        for dr, dc in DIRECTIONS:
            origin = (
                empty_row - dr,
                empty_col - dc
            )

            if inside_board(origin) and origin not in target:
                own_positions = set(target) - {empty_position}
                own_positions.add(origin)

                if candidate == P1:
                    return state_from_pieces(
                        p1_positions=own_positions,
                        turn=P1
                    )

                return state_from_pieces(
                    p2_positions=own_positions,
                    turn=P2
                )

    raise ValueError(
        "No se pudo construir el escenario de prueba"
    )

In [ ]:
state = one_move_from_winning(P1)

action, stats = minimax_alpha_beta(
    state,
    return_stats=True
)

print("Jugada elegida:", action)
print("Nodos visitados:", stats.visited_nodes)
print("Ramas podadas:", stats.pruned_branches)

Jugada elegida: ((5, 8), (5, 9))
Nodos visitados: 1
Ramas podadas: 0


# Función heurística
#### Definición de pesos de la función


In [21]:
PROGRESS_WEIGHT = 0.60
CAMP_WEIGHT = 0.40



Progress weigth mide el avance general de varias piezas, mientras que camp mide el avance para entrar al campamento rival. Ninguna distingue entre avanzar varias piezas en general o solamente si tienen el mismo progreso total. 

In [22]:
## DISTANCIA NORMALIZADA ES LA MÁXIMA DISTANCIA = 18
def piece_progress(position, candidate):
    """ Calcula el progreso de una pieza hacia el campamento contrincante"""
    row, col = position
    maximum_distance = 2*(BOARD_SIZE - 1)  # Distancia máxima posible en el tablero

    if candidate == P1:
        return (row + col) / maximum_distance

    return(
        ( (BOARD_SIZE - 1 - row) + (BOARD_SIZE - 1 - col) ) / maximum_distance
    )

In [31]:
check(
    "P1 en la esquina inicial tiene progreso 0",
    piece_progress((0, 0), P1) == 0
)

check(
    "P1 en el centro tiene progreso 0.5",
    piece_progress((4, 5), P1) == 0.5
)

check(
    "P1 en la esquina objetivo tiene progreso 1",
    piece_progress((9, 9), P1) == 1
)
check(
    "P2 en la esquina inicial tiene progreso 0",
    piece_progress((9, 9), P2) == 0
)

check(
    "P2 en la esquina objetivo tiene progreso 1",
    piece_progress((0, 0), P2) == 1
)

PASS  P1 en la esquina inicial tiene progreso 0
PASS  P1 en el centro tiene progreso 0.5
PASS  P1 en la esquina objetivo tiene progreso 1
PASS  P2 en la esquina inicial tiene progreso 0
PASS  P2 en la esquina objetivo tiene progreso 1


True

## Función del progreso de todas las piezas


In [32]:
def total_progress(state, candidate):
    """Suma el progreso de todas las piezas de candidate."""
    total = 0.0

    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if state.board[row][col] == candidate:
                total += piece_progress((row, col), candidate)

    return total

### Función para comparar el progreso de los jugadores

In [ ]:
def progress_balance(state):
    """Compara el progreso total de P1 contra el de P2."""
    p1_progress = total_progress(state, P1)
    p2_progress = total_progress(state, P2)

    return (p1_progress - p2_progress) / 15
## se normaliza para que los valores estén entre -1 y 1, ya que el progreso máximo de un jugador es 15 (todas sus piezas en el campamento contrario).